# Phase 4 — Canonical retrieval corpus

This notebook converts the signed Phase 3.1 result into Devoteam's controlled
retrieval corpus. It builds the document/reference catalogues, filtering
metadata, canonical redacted pages, and stable page-level chunks.

Run **Runtime → Run all**. Do not unzip the package manually. This phase makes
no OCR, embedding, or LLM calls and does not modify source or Phase 3 files.


## 1. Mount the known clean project


In [ ]:
import hashlib, json, os, subprocess, sys, zipfile
from pathlib import Path

EXPECTED_PACKAGE_SHA256 = "30e3bc167808edbc43097d67775f64dc974980031eca3b42cf32730fd42f7666"
PACKAGE_FILENAME = "PHASE_4_CANONICAL_CORPUS_PACKAGE.zip"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/Devoteam internship/Devoteam_AI_CLEAN_PIPELINE")
assert (PROJECT_ROOT / "config" / "project.yaml").exists(), f"Clean project not found: {PROJECT_ROOT}"
PHASE3_1_ROOT = PROJECT_ROOT / "data" / "extracted" / "20260714T154731Z_129ff982c8" / "phase3_1_targeted_repair_v1"
assert (PHASE3_1_ROOT / "_SUCCESS.json").exists(), "Signed Phase 3.1 result is missing"
PACKAGE_PATH = PROJECT_ROOT / PACKAGE_FILENAME
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
print(f"Project root: {PROJECT_ROOT}")


## 2. Verify and install the signed additive package


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_package_sha = file_sha256(PACKAGE_PATH)
assert actual_package_sha == EXPECTED_PACKAGE_SHA256, "Phase 4 package hash mismatch"

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    for member in archive.infolist():
        target = (PROJECT_ROOT / member.filename).resolve()
        assert str(target).startswith(str(PROJECT_ROOT.resolve()) + os.sep), f"Unsafe archive path: {member.filename}"
    package_manifest = json.loads(archive.read("PHASE_4_PACKAGE_MANIFEST.json"))
    assert package_manifest["pipeline_version"] == "phase4_corpus_v1"
    for entry in package_manifest["files"]:
        assert hashlib.sha256(archive.read(entry["path"])).hexdigest() == entry["sha256"]
    for member in archive.infolist():
        if member.is_dir() or member.filename == "PHASE_4_PACKAGE_MANIFEST.json":
            continue
        destination = PROJECT_ROOT / member.filename
        packaged_hash = hashlib.sha256(archive.read(member.filename)).hexdigest()
        if destination.exists():
            assert file_sha256(destination) == packaged_hash, f"Refusing to overwrite changed file: {member.filename}"
        else:
            archive.extract(member, PROJECT_ROOT)

print(f"Verified package SHA-256: {actual_package_sha}")
print("Phase 4 additive extension installed safely.")


## 3. Check dependencies and run the complete project test suite


In [ ]:
print("Installing/checking pinned Python packages — usually 1–3 minutes...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
        "-r", str(PROJECT_ROOT / "requirements" / "phase4.txt"),
    ],
    check=True,
)
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
print("Running foundation, snapshot, extraction, repair, and corpus tests...")
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests")],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
assert tests.returncode == 0, "Tests failed; Phase 4 did not start."
print("Complete project test suite passed.")


## 4. Build or resume the canonical corpus


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from devoteam_reference_ai.phase4_corpus import run_phase4

summary = run_phase4(
    project_root=PROJECT_ROOT,
    config_path=PROJECT_ROOT / "config" / "phase4_corpus.yaml",
    progress=print,
)
print(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True))


## 5. Verify hashes, schemas, security assertions, and final gate


In [ ]:
from devoteam_reference_ai.phase4_corpus import verify_phase4

run_root = Path(summary["run_root"])
verified = verify_phase4(run_root)
assert verified["documents_total"] == 134
assert verified["documents_retrieval_eligible"] == 132
assert verified["pages_canonical"] == 389
assert verified["pages_excluded"] == 19
assert verified["chunks_total"] == 1185
assert verified["references_total"] == 161
assert verified["missing_evidence_references"] == 21
assert verified["raw_text_output_columns"] == 0
assert verified["forbidden_workbook_fields_ingested"] == 0
assert verified["ocr_calls"] == 0
assert verified["embedding_calls"] == 0
assert verified["external_llm_calls"] == 0

print("PHASE 4: TECHNICAL PASS")
print(f"Canonical pages: {{verified['pages_canonical']}}")
print(f"Retrieval chunks: {{verified['chunks_total']}}")
print(f"Retrieval-ready documents: {{verified['documents_retrieval_eligible']}}/{{verified['documents_total']}}")
print(f"Workbook reference rows: {{verified['references_total']}}")
print(f"Excluded pages: {{verified['pages_excluded']}}")
print(f"Missing evidence references: {{verified['missing_evidence_references']}}")
print(f"Output: {{run_root}}")
print("Send this final block for senior review before Phase 5 indexing.")
